# Large Model — Global Scaling (scale first, then window)

Trains a single large model (h=256, l=8) with **global per-feature MinMax normalization**:  
the entire CSV is scaled first, then 32-step windows are sliced from the scaled series.  
This matches the Diffusion-TS / TimeGAN baseline protocol and produces scores that are  
directly comparable to published numbers.

**Key difference from previous runs**: `per_window_norm=False`  
- Previous: create windows → normalize each window independently → every window spans exactly [-1,1]  
- This run: normalize entire series globally (per feature) → create windows → windows sit at their  
  actual slice of the global [-1,1] range

Requires the project to be at `c:\Users\ameli\Desktop\TezBaselines\MyCode`.

In [1]:
import os, sys
from pathlib import Path

REPO_PATH = r'c:\Users\ameli\Desktop\TezBaselines\MyCode'
assert Path(REPO_PATH).exists(), f'Not found: {REPO_PATH}'

os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'Memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Working directory: {os.getcwd()}')

PyTorch : 2.11.0+cu126
CUDA    : True
GPU     : NVIDIA GeForce RTX 4070
Memory  : 12.9 GB
Working directory: c:\Users\ameli\Desktop\TezBaselines\MyCode


In [2]:
import wandb, os
os.environ['WANDB_API_KEY'] = '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'

_api_key = os.environ.get('WANDB_API_KEY')
if _api_key:
    wandb.login(key=_api_key, relogin=False)
else:
    wandb.login()

print('wandb version:', wandb.__version__)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\ameli\_netrc
wandb: Currently logged in as: a-meliksahdemir (a-meliksahdemir-bo-azi-i-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb version: 0.27.0


In [3]:
import torch, gc, importlib
from pathlib import Path

import train_with_mode as _twm
importlib.reload(_twm)
from train_with_mode import train

import evaluate_unified as _eu
importlib.reload(_eu)
from evaluate_unified import evaluate

DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
WANDB_PROJECT = 'diffusion-timeseries'
NUM_WORKERS   = 0
SEED          = 42
BATCH_SIZE    = 64

NUM_EPOCHS          = 2000
LR                  = 1e-4
EVAL_METRICS_EVERY  = 200
N_METRIC_ITERATIONS = 3

# ── Single experiment: large model, global scaling ─────────────────────────────
EXP = dict(
    name         = '256_8large_globalscale',
    group        = 'globalscale',
    hidden_dim   = 256,
    num_layers   = 8,
    fft_weight   = 1.0,
    trend_weight = 0.0,
    season_weight= 0.0,
    per_window_norm = False,   # <-- global scaling: normalize CSV first, then slice windows
)

print(f'Device  : {DEVICE}')
print(f'Run     : {EXP["name"]}  (h={EXP["hidden_dim"]} l={EXP["num_layers"]})')
print(f'Norm    : global per-feature MinMax  (per_window_norm=False)')
print(f'Epochs  : {NUM_EPOCHS}  |  eval_every={EVAL_METRICS_EVERY}  |  n_iter={N_METRIC_ITERATIONS}')
print()

ckpt_dir  = f"output/ckpt_{EXP['name']}"
best_ckpt = f"{ckpt_dir}/best_model.pt"

# ── Train ──────────────────────────────────────────────────────────────────────
try:
    train(
        mode                = 'decomposition',
        device              = DEVICE,
        hidden_dim          = EXP['hidden_dim'],
        num_layers          = EXP['num_layers'],
        num_epochs          = NUM_EPOCHS,
        lr                  = LR,
        batch_size          = BATCH_SIZE,
        num_workers         = NUM_WORKERS,
        seed                = SEED,
        use_wandb           = True,
        wandb_project       = WANDB_PROJECT,
        wandb_run_name      = EXP['name'],
        wandb_group         = EXP['group'],
        eval_metrics        = True,
        eval_metrics_every  = EVAL_METRICS_EVERY,
        n_metric_iterations = N_METRIC_ITERATIONS,
        img_pred_objective  = 'pred_x0',
        img_loss_type       = 'l1',
        fft_weight          = EXP['fft_weight'],
        trend_weight        = EXP['trend_weight'],
        season_weight       = EXP['season_weight'],
        checkpoint_dir      = ckpt_dir,
        finish_wandb        = False,
        per_window_norm     = EXP['per_window_norm'],
    )
except Exception:
    import traceback
    print(f'\n!!! TRAINING FAILED: {EXP["name"]}')
    traceback.print_exc()
    try:
        if wandb.run is not None: wandb.finish()
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

# ── Evaluate ───────────────────────────────────────────────────────────────────
if Path(best_ckpt).exists():
    print(f'\n--- Post-training evaluation: {EXP["name"]} ---')
    try:
        evaluate(
            mode                = 'decomposition',
            checkpoint_path     = best_ckpt,
            device              = DEVICE,
            num_samples         = 256,
            n_metric_iterations = N_METRIC_ITERATIONS,
            compute_context_fid = True,
            use_wandb           = True,
            wandb_project       = WANDB_PROJECT,
            wandb_run_name      = EXP['name'],
            wandb_group         = EXP['group'],
            output_dir          = ckpt_dir,
            hidden_dim          = EXP['hidden_dim'],
            num_layers          = EXP['num_layers'],
            per_window_norm     = EXP['per_window_norm'],
        )
    except Exception:
        import traceback
        print(f'\n!!! EVALUATION FAILED: {EXP["name"]}')
        traceback.print_exc()
        try:
            if wandb.run is not None: wandb.finish()
        except Exception:
            pass
else:
    print(f'   [skip eval] best_model.pt not found at {best_ckpt}')
    try:
        if wandb.run is not None: wandb.finish()
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\n' + '='*60)
print('  DONE')
print('='*60)

Device  : cuda
Run     : 256_8large_globalscale  (h=256 l=8)
Norm    : global per-feature MinMax  (per_window_norm=False)
Epochs  : 2000  |  eval_every=200  |  n_iter=3

Global seed set to 42


TRANSFORMER DIFFUSION MODEL - DECOMPOSITION MODE
{'model': {'hidden_dim': 256, 'num_layers': 8, 'num_heads': 8, 'ff_dim': 1024, 'dropout': 0.1, 'input_channels': 6, 'sequence_length': 32, 'learnable_pos_enc': True}, 'diffusion': {'num_timesteps': 1000, 'beta_start': 0.0001, 'beta_end': 0.02, 'noise_schedule': 'cosine', 'variance_type': 'fixed_large', 'gamma': 1.0}, 'training': {'batch_size': 64, 'learning_rate': 0.0001, 'num_epochs': 2000, 'warmup_steps': 100, 'weight_decay': 0.0001, 'gradient_clip_val': 1.0, 'lr_scheduler_type': 'cosine', 'checkpoint_dir': 'output/ckpt_256_8large_globalscale', 'log_dir': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\output\\logs', 'save_every_n_epochs': 250, 'validate_every_n_epochs': 10}, 'sampling': {'sampler_type': 'ddim', 'num_sampling_steps': 500, 'eta': 0.0, 'batch_size': 16, 'output_dir': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\output\\generated_samples'}, 'data': {'data_path': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\dat

context_fid,▅▂▇▆▆▄▃█▇▄▁
correlational_score,██▁▁▁▂▂▂▄▂▁
disc_score,█▄▄▅▅▂▅▄█▃▁
disc_score_std,▂▅▇▆▂█▂▆▄▄▁
epoch,▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███████
fdds,▂▁▄▃▇▅█▆▅▅▄
fft_loss,▆█▆▅▄▃▄▃▂▂▄▃▂▄▃▁▁▃▂▃▃▂▂▂▃▂▁▃▂▁▁▂▃▂▄▂▂▃▂▃
lr,█████████▇▇▇▆▆▆▆▆▆▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
pred_mae,█▁▂▁▂▂▂▂▄▂▁
pred_mae_std,▃▆▁▂█▃▄▄▅▆▅
+4,...


   W&B run finished.

Evaluation complete! (decomposition mode)
Results saved to: output/ckpt_256_8large_globalscale

  DONE
